# ハエの脳を触る — 最初の一歩

このノートは 2 つのコネクトーム(神経回路の完全配線図)を扱います。

| | hemibrain | FlyWire (FAFB) |
|---|---|---|
| 中身 | 成虫の脳の中心部を電顕で切って再構築 | 成虫メス 1 匹の脳まるごと |
| ニューロン | 約 2.5 万 (traced: 21,739) | 約 13.9 万 |
| ここにあるデータ | 接続行列 355 万本 | ニューロン注釈 13.9 万件 |

まずは実物のニューロンを 3D で見て、それから配線を数えます。

In [ ]:
import pandas as pd
import navis
import plotly.io as pio

pio.renderers.default = "notebook_connected"
navis.set_pbars(jupyter=False)
DATA = "../data/hemibrain/exported-traced-adjacencies-v1.2"
print(navis.__version__)

## 1. 実物のニューロンを 3D で見る

navis に同梱されている hemibrain の嗅覚投射ニューロン (DA1 uPN) の骨格データ。

In [ ]:
nl = navis.example_neurons(5)
brain = navis.example_volume("neuropil")
brain.color = (250, 250, 250, 0.08)
navis.plot3d([nl, brain], backend="plotly", width=900, height=650)

## 2. 配線表を読み込む

`weight` = 2 つのニューロン間のシナプス数。

In [ ]:
neurons = pd.read_csv(f"{DATA}/traced-neurons.csv")
conns = pd.read_csv(f"{DATA}/traced-total-connections.csv")
print(f"{len(neurons):,} ニューロン / {len(conns):,} 接続 / {conns['weight'].sum():,} シナプス")
neurons.head()

## 3. キノコ体出力ニューロン (MBON) の上流を調べる

MBON は「学習した匂いの好き嫌い」を出力する有名なニューロン。誰が入力しているか見る。

In [ ]:
name = neurons.set_index("bodyId")["type"]
mbon_ids = set(neurons[neurons["type"].fillna("").str.startswith("MBON")]["bodyId"])
print(f"MBON: {len(mbon_ids)} 細胞")

up = conns[conns["bodyId_post"].isin(mbon_ids)].copy()
up["partner"] = up["bodyId_pre"].map(name)
up.groupby("partner")["weight"].sum().nlargest(20)

## 4. ネットワークとして扱う

強い接続だけ残して有向グラフにし、2 つの細胞型の間の最短経路を探す。

In [ ]:
import networkx as nx

strong = conns[conns["weight"] >= 10]
G = nx.from_pandas_edgelist(strong, "bodyId_pre", "bodyId_post",
                            edge_attr="weight", create_using=nx.DiGraph)
print(G)

src = neurons[neurons["type"] == "DA1_lPN"]["bodyId"].iloc[0]   # 嗅覚(フェロモン)入力
dst = neurons[neurons["type"] == "MBON01"]["bodyId"].iloc[0]    # 学習出力
path = nx.shortest_path(G, src, dst)
[f"{i} ({name.get(i)})" for i in path]

## 5. FlyWire 全脳のカタログ

In [ ]:
fw = pd.read_csv("../data/flywire/flywire_783_annotations.tsv", sep="	", low_memory=False)
print(f"{len(fw):,} ニューロン")
fw["super_class"].value_counts()

In [ ]:
fw["top_nt"].value_counts()

## 次にやると面白いこと

- `traced-roi-connections.csv` を使って脳領域ごとの配線を見る
- 特定の細胞型だけのサブネットワークを描く (`navis.plot3d` + neuPrint から骨格取得)
- FlyWire の実シナプスデータを取る → README の「FlyWire の接続データ」参照
- https://codex.flywire.ai で root_id を検索してブラウザ上で 3D 表示